# Re-ID clustering experiment (M2 gate)\nRuntime -> Change runtime type -> **T4 GPU**, then Run all. Upload `reid_experiment.zip` when Cell 1 prompts.

In [ ]:
# Cell 1 — upload reid_experiment.zip when prompted, then run all
from google.colab import files
up = files.upload()  # choose reid_experiment.zip
import zipfile, os
zipfile.ZipFile(list(up)[0]).extractall("data")
print(len(os.listdir("data/crops")), "crops")

In [ ]:
# Cell 2 — DINOv2 embeddings + clustering
import torch, glob, numpy as np
from PIL import Image
import torchvision.transforms as T
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(dev).eval()
tf = T.Compose([T.Resize((224,112)), T.Pad((56,0)), T.ToTensor(),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
files_ = sorted(glob.glob("data/crops/*.jpg"))
embs = []
with torch.no_grad():
    for i in range(0, len(files_), 64):
        batch = torch.stack([tf(Image.open(f).convert("RGB")) for f in files_[i:i+64]]).to(dev)
        embs.append(model(batch).cpu().numpy())
X = np.concatenate(embs)
X = X / np.linalg.norm(X, axis=1, keepdims=True)
np.save("emb_dinov2.npy", X)
print("embeddings:", X.shape)

In [ ]:
# Cell 3 — cluster + montage (tune THR if needed: higher = fewer clusters)
import cv2, numpy as np, glob
from sklearn.cluster import AgglomerativeClustering
X = np.load("emb_dinov2.npy"); files_ = sorted(glob.glob("data/crops/*.jpg"))
for THR in (0.35, 0.45, 0.55):
    lab = AgglomerativeClustering(n_clusters=None, distance_threshold=THR,
          metric="cosine", linkage="average").fit(X).labels_
    sizes = np.bincount(lab)
    big = [c for c in np.argsort(-sizes) if sizes[c] >= 8]
    print(f"THR={THR}: {lab.max()+1} clusters, {len(big)} with >=8 crops")
    rows = []
    for c in big[:20]:
        idx = [i for i in range(len(files_)) if lab[i]==c]
        idx = idx[::max(1,len(idx)//10)][:10]
        tiles = []
        for i in idx:
            im = cv2.imread(files_[i]); s = 110/im.shape[0]
            tiles.append(cv2.resize(im,(max(16,int(im.shape[1]*s)),110)))
        w = sum(t.shape[1] for t in tiles)+len(tiles)*3+60
        cv = np.zeros((116,w,3),np.uint8)
        cv2.putText(cv,f"C{c}:{sizes[c]}",(2,60),0,0.5,(255,255,255),1)
        x = 60
        for t in tiles: cv[3:113,x:x+t.shape[1]]=t; x+=t.shape[1]+3
        rows.append(cv)
    wm = max(r.shape[1] for r in rows)
    rows = [np.pad(r,((0,0),(0,wm-r.shape[1]),(0,0))) for r in rows]
    cv2.imwrite(f"clusters_dinov2_thr{int(THR*100)}.jpg", np.vstack(rows))
print("wrote cluster montages")

In [ ]:
# Cell 4 — (optional) OSNet re-ID embeddings, same clustering
!pip -q install git+https://github.com/KaiyangZhou/deep-person-reid.git
import torchreid, torch, glob, numpy as np
from PIL import Image
import torchvision.transforms as T
dev = "cuda" if torch.cuda.is_available() else "cpu"
m = torchreid.models.build_model("osnet_x1_0", num_classes=1, pretrained=True).to(dev).eval()
tf = T.Compose([T.Resize((256,128)), T.ToTensor(),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
files_ = sorted(glob.glob("data/crops/*.jpg"))
embs = []
with torch.no_grad():
    for i in range(0, len(files_), 64):
        b = torch.stack([tf(Image.open(f).convert("RGB")) for f in files_[i:i+64]]).to(dev)
        embs.append(m(b).cpu().numpy())
X = np.concatenate(embs); X = X/np.linalg.norm(X,axis=1,keepdims=True)
np.save("emb_dinov2.npy", X)  # reuse Cell 3 to montage: rerun it after this
print("OSNet embeddings:", X.shape, "- now rerun Cell 3")

In [ ]:
# Cell 5 — download results
import zipfile, glob
with zipfile.ZipFile("results.zip","w") as z:
    for f in glob.glob("clusters_dinov2_*.jpg"): z.write(f)
from google.colab import files
files.download("results.zip")